In [1]:
from pathlib import Path
import shutil
import yaml


# ============================================================
# НАСТРОЙКИ
# ============================================================

# Путь к исходному датасету
DATASET_PATH = Path(r"C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\FYP_MOCSdataset.v3i.yolo26")

# Папка для результата
OUTPUT_PATH = DATASET_PATH.parent / f"{DATASET_PATH.name}_bbox"

# Поддерживаемые изображения
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"
}


# ============================================================
# ПАПКИ
# ============================================================

IMAGES_DIR = DATASET_PATH / "images"
LABELS_DIR = DATASET_PATH / "labels"

OUT_IMAGES_DIR = OUTPUT_PATH / "images"
OUT_LABELS_DIR = OUTPUT_PATH / "labels"

OUT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
OUT_LABELS_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# ПРОВЕРКА
# ============================================================

if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Датасет не найден: {DATASET_PATH}")

if not IMAGES_DIR.exists():
    raise FileNotFoundError(f"Не найдена папка images: {IMAGES_DIR}")

if not LABELS_DIR.exists():
    raise FileNotFoundError(f"Не найдена папка labels: {LABELS_DIR}")


# ============================================================
# ЧТЕНИЕ DATA.YAML
# ============================================================

yaml_path = DATASET_PATH / "data.yaml"

if not yaml_path.exists():
    yaml_path = DATASET_PATH / "dataset.yaml"

if not yaml_path.exists():
    raise FileNotFoundError(
        "Не найден data.yaml или dataset.yaml"
    )

with open(yaml_path, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

names = data.get("names", [])

print("Классы исходного датасета:")

if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=lambda x: int(x))]

for i, name in enumerate(names):
    print(f"  {i}: {name}")


# ============================================================
# ФУНКЦИЯ КОНВЕРТАЦИИ SEGMENTATION -> BBOX
# ============================================================

def segmentation_to_bbox(line):
    """
    YOLO segmentation:

    class x1 y1 x2 y2 x3 y3 ...

    -> YOLO bbox:

    class x_center y_center width height
    """

    parts = line.strip().split()

    if len(parts) < 7:
        return None

    try:
        class_id = int(float(parts[0]))
        coords = list(map(float, parts[1:]))
    except ValueError:
        return None

    # Должно быть чётное количество координат
    if len(coords) < 6 or len(coords) % 2 != 0:
        return None

    xs = coords[0::2]
    ys = coords[1::2]

    # Ограничиваем координаты диапазоном [0, 1]
    xs = [max(0.0, min(1.0, x)) for x in xs]
    ys = [max(0.0, min(1.0, y)) for y in ys]

    x_min = min(xs)
    x_max = max(xs)
    y_min = min(ys)
    y_max = max(ys)

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2

    width = x_max - x_min
    height = y_max - y_min

    return (
        class_id,
        x_center,
        y_center,
        width,
        height
    )


# ============================================================
# КОНВЕРТАЦИЯ
# ============================================================

image_files = [
    p for p in IMAGES_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

print(f"\nНайдено изображений: {len(image_files)}")

converted_images = 0
converted_labels = 0
empty_labels = 0
invalid_lines = 0
total_objects = 0


for image_path in image_files:

    # Относительный путь позволяет сохранить структуру
    relative_path = image_path.relative_to(IMAGES_DIR)

    # Копируем изображение
    output_image = OUT_IMAGES_DIR / relative_path
    output_image.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(image_path, output_image)

    # Ищем соответствующий label
    relative_label = relative_path.with_suffix(".txt")
    label_path = LABELS_DIR / relative_label
    output_label = OUT_LABELS_DIR / relative_label

    output_label.parent.mkdir(parents=True, exist_ok=True)

    # Если label отсутствует
    if not label_path.exists():
        empty_labels += 1
        continue

    new_lines = []

    with open(label_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:

        if not line.strip():
            continue

        bbox = segmentation_to_bbox(line)

        if bbox is None:
            invalid_lines += 1
            continue

        class_id, xc, yc, w, h = bbox

        new_line = (
            f"{class_id} "
            f"{xc:.6f} "
            f"{yc:.6f} "
            f"{w:.6f} "
            f"{h:.6f}"
        )

        new_lines.append(new_line)
        total_objects += 1

    # Если после конвертации объектов нет
    if not new_lines:
        if output_label.exists():
            output_label.unlink()

        # Удаляем соответствующее изображение
        if output_image.exists():
            output_image.unlink()

        empty_labels += 1
        continue

    # Записываем bbox-разметку
    with open(output_label, "w", encoding="utf-8") as f:
        f.write("\n".join(new_lines) + "\n")

    converted_images += 1
    converted_labels += 1


# ============================================================
# КОПИРУЕМ ОТДЕЛЬНЫЕ LABEL, ЕСЛИ ЕСТЬ ОШИБОЧНЫЕ / ORPHAN
# НЕ НУЖНО — ОСТАВЛЯЕМ ТОЛЬКО LABEL, СООТВЕТСТВУЮЩИЕ IMAGES
# ============================================================

valid_output_images = {
    p.relative_to(OUT_IMAGES_DIR).with_suffix(".txt")
    for p in OUT_IMAGES_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
}

for label_path in OUT_LABELS_DIR.rglob("*.txt"):

    relative_label = label_path.relative_to(OUT_LABELS_DIR)

    if relative_label not in valid_output_images:
        label_path.unlink()


# ============================================================
# СОЗДАЁМ НОВЫЙ DATA.YAML
# ============================================================

new_data = {
    "train": "images",
    "val": "images",
    "test": "images",
    "nc": len(names),
    "names": names
}

output_yaml = OUTPUT_PATH / "data.yaml"

with open(output_yaml, "w", encoding="utf-8") as f:
    yaml.dump(
        new_data,
        f,
        allow_unicode=True,
        sort_keys=False
    )


# ============================================================
# СТАТИСТИКА
# ============================================================

print("\n" + "=" * 60)
print("КОНВЕРТАЦИЯ ЗАВЕРШЕНА")
print("=" * 60)

print(f"Исходный датасет:     {DATASET_PATH}")
print(f"Результат:            {OUTPUT_PATH}")
print()
print(f"Изображений найдено:  {len(image_files)}")
print(f"Изображений сохранено:{converted_images}")
print(f"Label-файлов создано: {converted_labels}")
print(f"Объектов конвертировано: {total_objects}")
print(f"Пустых/проблемных:    {empty_labels}")
print(f"Некорректных строк:   {invalid_lines}")

print("\nСтруктура результата:")
print(f"  {OUTPUT_PATH}")
print(f"  ├── images/")
print(f"  ├── labels/")
print(f"  └── data.yaml")

print("\nКлассы:")
for i, name in enumerate(names):
    print(f"  {i}: {name}")

Классы исходного датасета:
  0: Bulldozer
  1: Concrete mixer
  2: Crane
  3: Excavator
  4: Hanging head
  5: Loader
  6: Other vehicle
  7: Pile driving
  8: Pump truck
  9: Roller
  10: Static crane
  11: Truck
  12: Worker

Найдено изображений: 9238

КОНВЕРТАЦИЯ ЗАВЕРШЕНА
Исходный датасет:     C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\FYP_MOCSdataset.v3i.yolo26
Результат:            C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\FYP_MOCSdataset.v3i.yolo26_bbox

Изображений найдено:  9238
Изображений сохранено:9238
Label-файлов создано: 9238
Объектов конвертировано: 53958
Пустых/проблемных:    0
Некорректных строк:   0

Структура результата:
  C:\Users\Ева\Desktop\Хакатон\data_preproc\datasets\FYP_MOCSdataset.v3i.yolo26_bbox
  ├── images/
  ├── labels/
  └── data.yaml

Классы:
  0: Bulldozer
  1: Concrete mixer
  2: Crane
  3: Excavator
  4: Hanging head
  5: Loader
  6: Other vehicle
  7: Pile driving
  8: Pump truck
  9: Roller
  10: Static crane
  11: Truck
  12: Wor